# 🍕 Food Delivery Data Analysis
### College Project | Python Data Analysis
---
**Topics Covered:**
- Data Loading & Inspection
- Data Cleaning & Preprocessing
- Exploratory Data Analysis (EDA)
- 12+ Visualizations
- Statistical Analysis
- Business Insights

## 0. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

print('✅ Libraries imported successfully!')

## 1. Generate & Load Dataset

In [ ]:
# Generate dataset if not present
if not os.path.exists('data/food_delivery_data.csv'):
    import generate_dataset

df = pd.read_csv('data/food_delivery_data.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

In [ ]:
print('Column Names:', list(df.columns))
print('\nData Types:')
print(df.dtypes)

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
print(f'\nDuplicate Rows: {df.duplicated().sum()}')

## 2. Data Cleaning & Preprocessing

In [ ]:
# Fill missing ratings (cancelled orders) with 0
df['Customer_Rating'].fillna(0, inplace=True)

# Convert date to datetime
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

# Create derived columns
df['Month_Num']    = df['Order_Date'].dt.month
df['Is_Weekend']   = df['Day_of_Week'].isin(['Saturday', 'Sunday']).astype(int)
df['Is_Delivered'] = (df['Order_Status'] == 'Delivered').astype(int)
df['Discount_Pct'] = round((df['Discount'] / df['Order_Value']) * 100, 2)

print('✅ Data cleaning complete!')
df.head()

## 3. Descriptive Statistics

In [ ]:
numeric_cols = ['Order_Value', 'Discount', 'Final_Amount',
                'Delivery_Time_min', 'Customer_Rating', 'Delivery_Fee']
df[numeric_cols].describe().round(2)

In [ ]:
print('Order Status Distribution:')
print(df['Order_Status'].value_counts())
print('\nTop 5 Restaurants:')
print(df['Restaurant_Name'].value_counts().head())

## 4. Visualizations

### 4.1 Order Status Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
status_counts = df['Order_Status'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#f39c12']
ax.pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
       colors=colors, startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title('Order Status Distribution', fontsize=16, fontweight='bold')
plt.show()

### 4.2 Orders per Restaurant

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
rest_counts = df['Restaurant_Name'].value_counts()
bars = ax.bar(rest_counts.index, rest_counts.values,
              color=sns.color_palette('Set2', len(rest_counts)))
ax.bar_label(bars, padding=3)
ax.set_title('Number of Orders per Restaurant', fontsize=15, fontweight='bold')
ax.set_xlabel('Restaurant'); ax.set_ylabel('Orders')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

### 4.3 Revenue by Cuisine

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
cuisine_rev = df.groupby('Cuisine_Type')['Final_Amount'].sum().sort_values()
bars = ax.barh(cuisine_rev.index, cuisine_rev.values,
               color=sns.color_palette('coolwarm', len(cuisine_rev)))
ax.bar_label(bars, fmt='₹%.0f', padding=4, fontsize=9)
ax.set_title('Total Revenue by Cuisine Type', fontsize=15, fontweight='bold')
ax.set_xlabel('Total Revenue (₹)')
plt.tight_layout(); plt.show()

### 4.4 Monthly Order Trend

In [ ]:
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
monthly = df.groupby('Month').size().reindex(month_order).dropna()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(monthly.index, monthly.values, marker='o', linewidth=2.5,
        color='#3498db', markersize=8)
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color='#3498db')
ax.set_title('Monthly Order Trend', fontsize=15, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Number of Orders')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

### 4.5 Orders by Hour of Day

In [ ]:
hourly = df.groupby('Hour').size()
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(hourly.index, hourly.values,
       color=['#e74c3c' if h in [12,13,19,20,21] else '#3498db' for h in hourly.index])
ax.set_title('Orders by Hour of Day  (Red = Peak Hours)', fontsize=15, fontweight='bold')
ax.set_xlabel('Hour (24h)'); ax.set_ylabel('Number of Orders')
ax.set_xticks(range(8,24))
plt.tight_layout(); plt.show()

### 4.6 Delivery Time Distribution

In [ ]:
delivered = df[df['Order_Status'] == 'Delivered']
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(delivered['Delivery_Time_min'], bins=25, color='#9b59b6',
        edgecolor='white', alpha=0.75, density=True)
delivered['Delivery_Time_min'].plot.kde(ax=ax, color='#2c3e50', linewidth=2.5)
ax.axvline(delivered['Delivery_Time_min'].mean(), color='#e74c3c',
           linestyle='--', linewidth=2,
           label=f"Mean = {delivered['Delivery_Time_min'].mean():.1f} min")
ax.set_title('Delivery Time Distribution', fontsize=15, fontweight='bold')
ax.set_xlabel('Delivery Time (minutes)'); ax.set_ylabel('Density')
ax.legend(); plt.tight_layout(); plt.show()

### 4.7 Delivery Time vs Traffic (Box Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
order = ['Low', 'Medium', 'High']
traffic_data = [delivered[delivered['Traffic_Level'] == t]['Delivery_Time_min']
                for t in order]
bp = ax.boxplot(traffic_data, labels=order, patch_artist=True,
                medianprops=dict(color='red', linewidth=2))
for patch, color in zip(bp['boxes'], ['#2ecc71', '#f39c12', '#e74c3c']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_title('Delivery Time vs Traffic Level', fontsize=15, fontweight='bold')
ax.set_xlabel('Traffic Level'); ax.set_ylabel('Delivery Time (minutes)')
plt.tight_layout(); plt.show()

### 4.8 Payment Method Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
pay_counts = df['Payment_Method'].value_counts()
ax.pie(pay_counts, labels=pay_counts.index, autopct='%1.1f%%',
       startangle=90, colors=sns.color_palette('pastel'),
       wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2))
ax.set_title('Payment Method Distribution (Donut)', fontsize=15, fontweight='bold')
plt.show()

### 4.9 Average Rating by Restaurant

In [ ]:
avg_rating = (df[df['Customer_Rating'] > 0]
              .groupby('Restaurant_Name')['Customer_Rating']
              .mean().sort_values(ascending=False))
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(avg_rating.index, avg_rating.values,
              color=sns.color_palette('RdYlGn', len(avg_rating)))
ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=10)
ax.set_ylim(0, 5.5)
ax.axhline(avg_rating.mean(), linestyle='--', color='navy', linewidth=1.5,
           label=f'Avg = {avg_rating.mean():.2f}')
ax.set_title('Average Customer Rating by Restaurant', fontsize=15, fontweight='bold')
ax.set_ylabel('Rating (/ 5)'); ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

### 4.10 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
corr_cols = ['Order_Value','Discount','Final_Amount','Delivery_Fee',
             'Delivery_Time_min','Customer_Rating','Is_Weekend','Is_Delivered']
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

### 4.11 Revenue by City

In [ ]:
city_rev = df.groupby('City')['Final_Amount'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(city_rev.index, city_rev.values,
              color=sns.color_palette('tab10', len(city_rev)))
ax.bar_label(bars, fmt='₹%.0f', padding=3, fontsize=8, rotation=45)
ax.set_title('Total Revenue by City', fontsize=15, fontweight='bold')
ax.set_xlabel('City'); ax.set_ylabel('Revenue (₹)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

### 4.12 Orders by Day of Week & Status

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df.groupby(['Day_of_Week','Order_Status']).size().unstack(fill_value=0)
day_counts = day_counts.reindex(day_order)
fig, ax = plt.subplots(figsize=(10, 5))
day_counts.plot(kind='bar', ax=ax, color=['#2ecc71','#e74c3c','#f39c12'],
                edgecolor='white')
ax.set_title('Orders by Day of Week & Status', fontsize=15, fontweight='bold')
ax.set_xlabel('Day'); ax.set_ylabel('Orders')
ax.legend(title='Status'); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

## 5. Statistical Analysis

In [ ]:
# T-Test: Is delivery time significantly different in Low vs High traffic?
low_t  = delivered[delivered['Traffic_Level'] == 'Low']['Delivery_Time_min']
high_t = delivered[delivered['Traffic_Level'] == 'High']['Delivery_Time_min']
t_stat, p_val = stats.ttest_ind(low_t, high_t)

print('T-Test — Delivery Time: Low vs High Traffic')
print(f'  t-statistic = {t_stat:.4f}')
print(f'  p-value     = {p_val:.6f}')
if p_val < 0.05:
    print('  ✅ Statistically significant difference (p < 0.05)')
else:
    print('  ❌ No significant difference')

In [ ]:
# Pearson Correlation
r, p = stats.pearsonr(delivered['Order_Value'], delivered['Delivery_Time_min'])
print(f'Pearson Correlation (Order Value vs Delivery Time):')
print(f'  r = {r:.4f},  p = {p:.4f}')

## 6. Business Insights & Conclusions

In [ ]:
cancellation_rate = (df['Order_Status'] == 'Cancelled').mean() * 100
peak_hour = df.groupby('Hour').size().idxmax()

print('=== KEY BUSINESS INSIGHTS ===')
print(f'1. Cancellation Rate    : {cancellation_rate:.1f}%')
print(f'2. Best Rated Restaurant: {avg_rating.idxmax()}  ({avg_rating.max():.2f} ⭐)')
print(f'3. Peak Ordering Hour   : {peak_hour}:00')
print(f'4. Busiest Day          : {df["Day_of_Week"].value_counts().idxmax()}')
print(f'5. Top Revenue City     : {city_rev.idxmax()}  (₹{city_rev.max():,.0f})')
print(f'6. Most Used Payment    : {df["Payment_Method"].value_counts().idxmax()}')
print(f'7. Avg Delivery Time    : {delivered["Delivery_Time_min"].mean():.1f} min')
print(f'8. Avg Customer Rating  : {delivered["Customer_Rating"].mean():.2f} / 5.0')
print(f'9. Total Revenue        : ₹{delivered["Final_Amount"].sum():,.2f}')
print(f'10. Traffic Impact      : +{high_t.mean() - low_t.mean():.1f} min (Low → High)')

---
## ✅ Conclusion

This analysis of **1,000 food delivery orders** revealed several key business insights:

- **Most orders are successfully delivered** (~80%), with a manageable cancellation rate.
- **Peak hours are 12–13 (lunch) and 19–21 (dinner)**, suggesting staffing should be optimized for these windows.
- **High traffic significantly increases delivery time**, which negatively impacts customer satisfaction.
- **UPI and digital payments dominate**, reflecting modern payment preferences.
- **Indian and Italian cuisines generate the most revenue**, making them ideal for promotional campaigns.
- **Weekends and Fridays** see higher order volumes — useful for targeted discounts.

These insights can help food delivery platforms improve operations, reduce cancellations, and increase customer satisfaction.